In [ ]:
!unzip dir.zip

Archive:  efdl_project_itmo-codex-fix-search-parameters-for-enumeration.zip
   creating: efdl_project_itmo-codex-fix-search-parameters-for-enumeration/
  inflating: __MACOSX/._efdl_project_itmo-codex-fix-search-parameters-for-enumeration  
  inflating: efdl_project_itmo-codex-fix-search-parameters-for-enumeration/informer_tool.py  
  inflating: __MACOSX/efdl_project_itmo-codex-fix-search-parameters-for-enumeration/._informer_tool.py  
  inflating: efdl_project_itmo-codex-fix-search-parameters-for-enumeration/uv.lock  
  inflating: __MACOSX/efdl_project_itmo-codex-fix-search-parameters-for-enumeration/._uv.lock  
  inflating: efdl_project_itmo-codex-fix-search-parameters-for-enumeration/test_llm_with_tools.py  
  inflating: __MACOSX/efdl_project_itmo-codex-fix-search-parameters-for-enumeration/._test_llm_with_tools.py  
  inflating: efdl_project_itmo-codex-fix-search-parameters-for-enumeration/pyproject.toml  
  inflating: __MACOSX/efdl_project_itmo-codex-fix-search-parameters-for-enume

In [ ]:
%cd dir/

In [ ]:
!cd dir

In [3]:
!ls

example.py		      llm_serve.sh	pyproject.toml		uv.lock
Informer2020		      optuna_module.py	README.md
informer_tool.py	      optuna_plots	test_llm_with_tools.py
llm_optimization_informer.py  __pycache__	test_optuna_module.py


In [ ]:
!uv add "optuna>=3.6.1"

In [ ]:
!uv sync

In [ ]:
pip install  "optuna>=3.6.1"

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
from optuna.pruners import MedianPruner
from optuna.samplers import TPESampler

from optuna_module import (
    InformerSearchSpace,
    save_study_visualizations,
    start_informer2020_optuna_search,
    suggest_start_informer2020_search_kwargs,
)



custom_search_space = InformerSearchSpace(
    seq_len_choices=(48, 96, 192, 336),
    label_len_choices=(24, 48, 96, 168),
    pred_len_choices=(24, 48, 96, 192),
    d_model_choices=(256, 512),
    n_heads_choices=(4, 8),
    e_layers_choices=(2,),
    d_layers_choices=(1,),
    d_ff_choices=(512, 1024),
    factor_choices=(1, 3),
    dropout=(0.01, 0.1),
    learning_rate=(5e-6, 2e-4),
    batch_size_choices=(16, 32),
    train_epochs=(4, 12),
    patience=(2, 4),
    s_layers_choices=("3,2,1",),
    attn_choices=("prob", "full"),
    embed_choices=("timeF", "fixed"),
    activation_choices=("gelu",),
    distil_options=(True, False),
    output_attention_options=(False, True),
    mix_options=(True, False),
    padding_options=(0,),
    lradj_choices=("type1",),
)

kwargs = suggest_start_informer2020_search_kwargs(
    n_trials=10,
    metric="val_loss",
    seed=42,
    fixed_parameters={
        "train_epochs": 4,
        "batch_size": 32,
    },
    search_space=custom_search_space,
    sampler=TPESampler(seed=42),
    pruner=MedianPruner(n_startup_trials=2, n_warmup_steps=0, interval_steps=1),
)

study = start_informer2020_optuna_search(**kwargs)
print(f"Best trial value: {study.best_value}")

[I 2025-11-21 22:12:19,363] A new study created in memory with name: no-name-bcbe21ba-e6be-43e9-b5b0-12435289bf61


  0%|          | 0/10 [00:00<?, ?it/s]

[I 2025-11-21 22:12:19,376] Trial 0 pruned. label_len cannot exceed seq_len
[I 2025-11-21 22:25:22,974] Trial 1 finished with value: 1.0477 and parameters: {'seq_len': 336, 'label_len': 24, 'pred_len': 48, 'd_model': 256, 'n_heads': 8, 'e_layers': 2, 'd_layers': 1, 'd_ff': 1024, 'factor': 3, 'dropout': 0.06331731119758383, 'learning_rate': 5.934530307791969e-06, 'batch_size': 16, 'train_epochs': 4, 'patience': 4, 's_layers': '3,2,1', 'attn': 'prob', 'embed': 'timeF', 'activation': 'gelu', 'distil': True, 'output_attention': True, 'mix': False, 'padding': 0, 'lradj': 'type1'}. Best is trial 1 with value: 1.0477.


In [ ]:
output_dir = Path("optuna_plots")
plot_files = save_study_visualizations(
    study,
    output_dir=output_dir,
    contour_params=["learning_rate", "dropout", "d_model"],
)
print("Saved study visualisations:")
for file_path in plot_files:
    print(f" - {file_path}")


In [ ]:
if plot_files:
    first_plot = plot_files[0]

values = [trial.value for trial in study.trials if trial.value is not None]
fig, ax = plt.subplots()
ax.plot(range(1, len(values) + 1), values, marker="o")
ax.set_title("Optimization history (matplotlib)")
ax.set_xlabel("Trial number")
ax.set_ylabel("Objective value")
ax.grid(True, alpha=0.3)

matplotlib_snapshot = output_dir / "optimization_history_matplotlib.png"
fig.tight_layout()
fig.savefig(matplotlib_snapshot)
plt.close(fig)

print(f"Matplotlib snapshot saved to {matplotlib_snapshot}")


In [ ]:
8